# 02 - GAN base (vanilla, MLP) entrenada con Antibacterial

Entrenamiento de la GAN base con las secuencias de **Antibacterial** (`Antibacterial_amps_clean.fasta`, 5-60 aa) y evaluación de lo generado frente a las secuencias reales. Esta GAN es la **línea base** de las comparaciones posteriores, por eso se usa la configuración por defecto `GANConfig()` (z = 100, MLP, lote 128, semilla 42) sin ajustar hiperparámetros.

Orden del notebook: entorno, prueba de rendimiento (GPU frente a CPU y backends de Keras) y elección de la configuración más rápida, datos, entrenamiento, evaluación y conclusiones descriptivas de la vanilla pura (secciones a-f) y, a continuación, la **GAN vanilla estabilizada** (sección g): la misma GAN con tres correcciones estándar para datos discretos, entrenada con el mismo protocolo y comparada con la pura.

> **Tiempo de ejecución aproximado.** El notebook entrena **dos** modelos de 200 épocas y repite la prueba de rendimiento reducida. Con torch en GPU (RTX 3060): prueba de rendimiento ~40 s, vanilla pura ~6-7 min (~1,9 s/época) y estabilizada ~7-8 min (~2,2 s/época), unos 15 min en total. En CPU, con ~3 s/época por modelo, serían ~10 min por modelo (~20 min de entrenamiento) más la prueba de rendimiento, es decir, unos 25 min.

La lógica vive en `src/amp_gan/` (`benchmark`, `runtime`, `experiment`, `evaluation`, `properties`); aquí solo se importa, se ejecuta y se dibuja. Las métricas de evaluación son: longitud, composición de aminoácidos, carga neta (pH 7,4) e hidrofobicidad media de Kyte-Doolittle (GRAVY). Las secuencias generadas se decodifican de dos formas: **sample** (se muestrea cada posición de su distribución; modo principal) y **argmax** (se toma el aminoácido más probable; modo secundario).

## 0. Preparación: rutas, semilla y versiones

In [ ]:
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

# Raíz del proyecto: se sube desde el directorio actual hasta encontrar `context/` y `src/` (sin rutas absolutas).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "context").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Importamos solo módulos que NO cargan Keras (el backend debe fijarse antes de importarlo).
from amp_gan import runtime
from amp_gan.benchmark import collect_benchmark, select_fastest

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROCESSED = ROOT / "context" / "data" / "processed"
TRAIN_FASTA = PROCESSED / "Antibacterial_amps_clean.fasta"
BENCHMARK_CSV = PROCESSED / "benchmark_results.csv"
BATCH_SIZE = 128
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("python    :", platform.python_version())
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("scipy     :", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("semilla   :", SEED)
print("keras ya importado en este kernel:", "keras" in sys.modules)

## a. Entorno

Versiones de las librerías de aprendizaje profundo (leídas de los metadatos del paquete, sin importarlas) y GPU que ve el sistema. Todavía no se ha importado Keras: su backend y su dispositivo se eligen en la sección siguiente.

In [ ]:
import importlib.metadata as md_
import os
import subprocess

def installed_version(*names):
    for name in names:
        try:
            return f"{md_.version(name)} (paquete {name})"
        except md_.PackageNotFoundError:
            pass
    return "no instalado"

for label, names in (("keras", ("keras", "keras-nightly")), ("tensorflow", ("tensorflow",)), ("torch", ("torch",))):
    print(f"{label:11s}: {installed_version(*names)}")
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20, check=True).stdout.strip()
    print("GPU (nvidia-smi):", gpu)
except Exception as exc:
    print("GPU (nvidia-smi): no disponible:", type(exc).__name__)

## b. Prueba de rendimiento: GPU frente a CPU y backends

Se mide el paso de entrenamiento de la GAN (mismo modelo, mismos datos, misma semilla) para cada configuración `(backend, dispositivo)` y varios tamaños de lote. Cada configuración se ejecuta en un subproceso independiente para que el backend y la GPU queden bien aislados. Para que el notebook dure aproximadamente un minuto se miden menos pasos que en la prueba completa de `python -m amp_gan.benchmark` (60 pasos medidos y 20 de calentamiento, frente a 200 y 20), de modo que las cifras son algo más ruidosas. El resultado se devuelve en memoria y no se escribe a disco.

In [ ]:
t0 = time.perf_counter()
bench = collect_benchmark(steps=60, warmup=20)
print(f"Prueba de rendimiento hecha en {time.perf_counter() - t0:.0f} s")

view = bench[["backend", "device", "batch_size", "status", "ms_per_step_median", "samples_per_s", "est_epoch_s", "error"]].copy()
view["error"] = view["error"].fillna("").str.slice(0, 90)
display(view.round(3).reset_index(drop=True))

Tiempo estimado por época (segundos; menos es mejor) por configuración y tamaño de lote. Las configuraciones que fallan no tienen barra: se marcan como *no disponible* en la leyenda y su motivo se resume debajo del gráfico.

In [ ]:
configs = list(dict.fromkeys(zip(bench["backend"], bench["device"])))
batch_sizes = sorted(bench["batch_size"].unique())
colors = {("tensorflow", "cpu"): "tab:blue", ("torch", "cpu"): "tab:green",
          ("torch", "gpu"): "tab:red", ("tensorflow", "gpu"): "tab:purple"}

fig, ax = plt.subplots(figsize=(10, 4.2))
width = 0.8 / len(configs)
for i, (backend, device) in enumerate(configs):
    sub = bench[(bench["backend"] == backend) & (bench["device"] == device)].set_index("batch_size")
    ok = (sub["status"] == "ok").any()
    label = f"{backend}/{device}" + ("" if ok else " (no disponible)")
    x = np.arange(len(batch_sizes)) + (i - (len(configs) - 1) / 2) * width
    vals = [sub.loc[b, "est_epoch_s"] if b in sub.index and sub.loc[b, "status"] == "ok" else np.nan for b in batch_sizes]
    bars = ax.bar(x, np.nan_to_num(vals), width=width, color=colors.get((backend, device), "gray"), label=label)
    for rect, v in zip(bars, vals):
        if np.isfinite(v):
            ax.text(rect.get_x() + rect.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(np.arange(len(batch_sizes)))
ax.set_xticklabels(batch_sizes)
ax.set_xlabel("Tamaño de lote")
ax.set_ylabel("Tiempo estimado por época (s)")
ax.set_title("Prueba de rendimiento de la GAN base: tiempo por época")
ax.legend(title="backend/dispositivo")
fig.tight_layout()
plt.show()

errors = bench[bench["status"] != "ok"].drop_duplicates(["backend", "device"])
for _, r in errors.iterrows():
    print(f"No disponible: {r['backend']}/{r['device']} -> {str(r['error'])[:200]}")

Se aplica la regla de decisión del proyecto para el tamaño de lote de entrenamiento (128): gana la configuración con menor tiempo estimado por época; si la segunda está a menos de un 10 % se declara empate y se prefiere la opción más simple. Después se fija el backend y el dispositivo elegidos **antes de importar Keras** y se compara con la decisión guardada en `benchmark_results.csv` (prueba completa).

> **Nota.** Keras lee el backend (`KERAS_BACKEND`) y TensorFlow/torch leen la visibilidad de la GPU (`CUDA_VISIBLE_DEVICES`) solo al importarse. Si el kernel ya hubiese importado Keras con otro backend, habría que reiniciarlo. Por eso hasta aquí no se ha importado Keras ni ninguna función que lo cargue.

In [ ]:
choice = select_fastest(bench, BATCH_SIZE)
print(f"Lote {BATCH_SIZE} (prueba reducida): {choice['message']}")
official = select_fastest(pd.read_csv(BENCHMARK_CSV), BATCH_SIZE)
print(f"Lote {BATCH_SIZE} (prueba completa, benchmark_results.csv): {official['message']}")
print("¿Misma configuración?", (choice["backend"], choice["device"]) == (official["backend"], official["device"]))

assert "keras" not in sys.modules, "Keras ya está importado: reinicia el kernel antes de elegir el backend."
runtime.set_backend(choice["backend"])
if choice["device"] == "cpu":
    runtime.use_cpu_only()
print(f"Configuración aplicada: backend={choice['backend']}, dispositivo={choice['device']}")

Ahora sí se importa Keras (a través de `describe_environment`) y se confirma el backend y el dispositivo realmente usados.

In [ ]:
env = runtime.describe_environment()
for k in ("keras_backend", "keras_devices", "gpu_visible", "cuda_visible_devices", "keras_version"):
    print(f"{k:22s}: {env[k]}")
assert env["keras_backend"] == choice["backend"]
assert env["gpu_visible"] == (choice["device"] == "gpu")

## c. Datos

Se cargan las secuencias limpias de Antibacterial (5-60 aa, 20 aminoácidos estándar, sin duplicados exactos) y se codifican en one-hot con relleno `PAD` hasta 60 posiciones.

In [ ]:
from amp_gan.encoding import SEQ_LEN, VOCAB_SIZE, load_encoded

X, df = load_encoded(TRAIN_FASTA)
print(f"secuencias: {len(df):,}")
print(f"X: forma {X.shape}, dtype {X.dtype}, tamaño {X.nbytes / 2**20:.0f} MiB")
print(f"seq_len = {SEQ_LEN}, vocabulario = {VOCAB_SIZE} (20 aa + PAD)")

## d. Entrenamiento

200 épocas con la configuración por defecto. Cada 20 épocas (y en la primera y la última) se generan 2000 secuencias con un ruido fijo y se evalúan en los dos modos de decodificación. La primera época incluye la construcción del modelo, y el tiempo de entrenamiento no incluye el de las evaluaciones.

In [ ]:
from amp_gan.experiment import run_gan_experiment
from amp_gan.gan import GANConfig

EPOCHS, SNAPSHOT_EVERY, N_EVAL = 200, 20, 2000
config = GANConfig()
print(config)

res = run_gan_experiment(X, df, config, epochs=EPOCHS, snapshot_every=SNAPSHOT_EVERY, n_eval=N_EVAL, eval_seed=SEED)
t = res.timing
print(f"Tiempo total: {t['total_s']:.1f} s (entrenamiento {t['train_s']:.1f} s + evaluaciones {t['eval_s']:.1f} s)")
print(f"Por época: media {t['mean_epoch_s']:.2f} s, mediana {t['median_epoch_s']:.2f} s")

Pérdidas medias por época. A la izquierda, el discriminador con lotes reales (D real) y falsos (D falso); a la derecha, el generador. La línea punteada marca ln 2 ≈ 0,693, el valor de la entropía cruzada cuando una salida vale 0,5. La pérdida del discriminador se dibuja en escala logarítmica para que se vea aunque sea muy pequeña.

In [ ]:
h = res.history
epochs_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(epochs_axis, h["d_loss_real"], label="D real", color="tab:blue")
axes[0].plot(epochs_axis, h["d_loss_fake"], label="D falso", color="tab:orange")
axes[0].set_yscale("log")
axes[0].set_title("Pérdida del discriminador (escala logarítmica)")
axes[1].plot(epochs_axis, h["g_loss"], label="G", color="tab:green")
axes[1].set_title("Pérdida del generador")
for ax in axes:
    ax.axhline(np.log(2), color="gray", ls=":", lw=1, label="ln 2")
    ax.set_xlabel("Época")
    ax.set_ylabel("Pérdida (entropía cruzada binaria)")
    ax.legend()
fig.tight_layout()
plt.show()

hist_df = pd.DataFrame({k: h[k] for k in ("d_loss_real", "d_loss_fake", "d_loss", "g_loss")}, index=epochs_axis)
display(pd.DataFrame({"primeras 5 épocas": hist_df.head(5).mean(), "últimas 5 épocas": hist_df.tail(5).mean()}).round(4))

## e. Evaluación frente a las secuencias reales

### Métricas por instantánea

Para cada instantánea y modo: fracción de secuencias válidas (aminoácidos estándar y 5-60 aa), únicas (entre las válidas) y nuevas (que no están en Antibacterial), y, para longitud, carga neta (pH 7,4) e hidrofobicidad (GRAVY): media, desviación típica y distancia de Wasserstein-1 (W1) entre la distribución generada y la real (las unidades son las de cada propiedad; 0 = idénticas). `composition_tv` es la distancia de variación total entre la composición media de aminoácidos generada y la real (0 = idéntica, 1 = disjuntas).

Se añaden dos filas de **suelo** (misma `n` y mismas métricas): `real_subset`, una submuestra aleatoria de secuencias reales frente al resto (lo mínimo que se puede esperar por el ruido del muestreo; su novedad es 1 por construcción) y `random_uniform`, secuencias aleatorias con aminoácidos y longitudes uniformes (5-60), como cota de "ningún aprendizaje".

In [ ]:
from amp_gan.evaluation import reference_floors
from amp_gan.properties import properties_table

floors = reference_floors(df, n=N_EVAL, seed=SEED)
floor_rows = pd.DataFrame([{"epoch": "suelo", "mode": name, **vals} for name, vals in floors.items()])
metrics = res.metrics.copy()
metrics["epoch"] = metrics["epoch"].astype(str)

cols = ["valid_fraction", "unique_fraction", "novel_fraction", "length_mean", "length_std", "length_wasserstein",
        "net_charge_mean", "net_charge_std", "net_charge_wasserstein",
        "hydrophobicity_mean", "hydrophobicity_std", "hydrophobicity_wasserstein", "composition_tv"]

def metrics_table(mode):
    sel = metrics[metrics["mode"] == mode]
    out = pd.concat([sel, floor_rows], ignore_index=True)
    out["fila"] = np.where(out["epoch"] == "suelo", out["mode"], "época " + out["epoch"])
    return out.set_index("fila")[cols]

props_real = properties_table(df)
ref_stats = pd.DataFrame({
    "media": props_real[["length", "net_charge", "hydrophobicity"]].mean(),
    "desv. típica": props_real[["length", "net_charge", "hydrophobicity"]].std(),
})
print("Referencia real (Antibacterial completo):")
display(ref_stats.round(3))
print("Modo principal: sample")
display(metrics_table("sample").round(3))

Modo secundario: **argmax** (aminoácido más probable en cada posición), con las mismas filas de suelo.

In [ ]:
display(metrics_table("argmax").round(3))

### Distribuciones: real frente a generado (época final)

Histogramas normalizados a fracción de secuencias, con los mismos intervalos y ejes para real y generado en cada panel. Arriba, modo sample; abajo, modo argmax. Solo se usan las secuencias generadas válidas.

In [ ]:
from amp_gan.evaluation import is_valid, profile_sequences

PROPS = [("length", "Longitud (aa)"), ("net_charge", "Carga neta (pH 7,4)"), ("hydrophobicity", "Hidrofobicidad (GRAVY)")]
valid_seqs = {m: [s for s in res.sequences[m] if is_valid(s)] for m in ("sample", "argmax")}
prof_gen = {m: profile_sequences(v) if v else None for m, v in valid_seqs.items()}
prof_real = profile_sequences(df["sequence"].tolist())

# Intervalos comunes por propiedad (real y ambos modos generados).
bin_edges = {}
for prop, _ in PROPS:
    allv = np.concatenate([prof_real.values(prop)] + [p.values(prop) for p in prof_gen.values() if p is not None])
    lo, hi = np.floor(allv.min()), np.ceil(allv.max())
    bin_edges[prop] = np.arange(lo - 0.5, hi + 1.5) if prop == "length" else np.linspace(lo, hi, 41)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex="col")
for r, mode in enumerate(("sample", "argmax")):
    for c, (prop, xlabel) in enumerate(PROPS):
        ax = axes[r, c]
        real = prof_real.values(prop)
        ax.hist(real, bins=bin_edges[prop], weights=np.full(real.size, 1 / real.size), color="tab:blue", alpha=0.45,
                label=f"real (n={real.size:,})")
        p = prof_gen[mode]
        if p is not None:
            g = p.values(prop)
            ax.hist(g, bins=bin_edges[prop], weights=np.full(g.size, 1 / g.size), histtype="step", lw=1.8, color="tab:red",
                    label=f"generado, {mode} (n={g.size:,})")
        ax.set_title(f"{xlabel} - {mode}")
        ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel("Fracción de secuencias")
        ax.set_ylim(0, ax.get_ylim()[1] * 1.3)  # hueco para la leyenda
        ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()

### Composición de aminoácidos

Fracción media de cada aminoácido (media de las fracciones por secuencia) en las secuencias reales y en las generadas válidas, ordenados por abundancia en las reales.

In [ ]:
from amp_gan.data import STANDARD_ALPHABET

comp = pd.DataFrame({"real": prof_real.composition}, index=list(STANDARD_ALPHABET))
for mode in ("sample", "argmax"):
    comp[mode] = prof_gen[mode].composition if prof_gen[mode] is not None else np.nan
comp = comp.sort_values("real", ascending=False)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(comp))
w = 0.27
for i, (name, color) in enumerate((("real", "tab:blue"), ("sample", "tab:red"), ("argmax", "tab:orange"))):
    label = "real" if name == "real" else f"generado, {name}"
    ax.bar(x + (i - 1) * w, comp[name] * 100, width=w, color=color, label=label)
ax.set_xticks(x)
ax.set_xticklabels(comp.index)
ax.set_xlabel("Aminoácido (ordenados por abundancia en las secuencias reales)")
ax.set_ylabel("Composición media (%)")
ax.set_title("Composición media de aminoácidos: real frente a generado (época final)")
ax.legend()
fig.tight_layout()
plt.show()

### Evolución de las métricas durante el entrenamiento

Métricas de cada instantánea (línea continua: sample; discontinua: argmax). Las líneas horizontales son los suelos `real_subset` (punteada gris) y, cuando cabe en la escala, `random_uniform` (punteada roja); en la media de longitud, la línea gris es la media real.

In [ ]:
panels = [
    ("unique_fraction", "Fracción de secuencias únicas", None),
    ("length_wasserstein", "W1 de longitud (aa)", None),
    ("net_charge_wasserstein", "W1 de carga neta", None),
    ("hydrophobicity_wasserstein", "W1 de hidrofobicidad", None),
    ("composition_tv", "Variación total de composición", None),
    ("length_mean", "Longitud media (aa)", float(props_real["length"].mean())),
]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, (col, title, real_value) in zip(axes.ravel(), panels):
    for mode, ls in (("sample", "-"), ("argmax", "--")):
        sel = res.metrics[res.metrics["mode"] == mode]
        ax.plot(sel["epoch"], sel[col], ls, marker="o", ms=4, label=mode)
    if real_value is not None:
        ax.axhline(real_value, color="gray", ls=":", label="real")
    else:
        ax.axhline(floors["real_subset"][col], color="gray", ls=":", label="suelo real_subset")
        ax.axhline(floors["random_uniform"][col], color="tab:red", ls=":", lw=1, label="suelo random_uniform")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.set_ylabel(title)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Secuencias generadas de ejemplo

Las diez primeras secuencias de cada modo (época final) con su longitud, carga neta e hidrofobicidad (las no válidas aparecen sin propiedades).

In [ ]:
def example_table(mode, k=10):
    rows = []
    for s in res.sequences[mode][:k]:
        if is_valid(s):
            p = profile_sequences([s])
            rows.append({"sequence": s, "length": len(s), "net_charge": p.net_charge[0], "hydrophobicity": p.hydrophobicity[0]})
        else:
            rows.append({"sequence": s if s else "(vacía)", "length": len(s), "net_charge": np.nan, "hydrophobicity": np.nan})
    return pd.DataFrame(rows).round(2)

pd.set_option("display.max_colwidth", 80)
print("Modo sample:")
display(example_table("sample"))
print("Modo argmax:")
display(example_table("argmax"))

## f. Conclusiones (descriptivas)

El texto siguiente se genera a partir de las cifras de esta ejecución y se limita a describir lo medido: qué se observa y dónde queda lejos de los datos reales. No interpreta las causas ni anticipa mejoras.

In [ ]:
from collections import Counter

fin = res.metrics[res.metrics["epoch"] == EPOCHS].set_index("mode")
fl = floors["real_subset"]
ru = floors["random_uniform"]
print(f"Entrenamiento: {EPOCHS} épocas, backend {env['keras_backend']} en {choice['device']}, "
      f"{t['mean_epoch_s']:.2f} s/época de media ({t['total_s']:.0f} s en total).")
first, last = hist_df.head(5).mean(), hist_df.tail(5).mean()
print(f"Pérdidas medias (primeras 5 -> últimas 5 épocas): D real {first['d_loss_real']:.3f} -> {last['d_loss_real']:.3f}; "
      f"D falso {first['d_loss_fake']:.3f} -> {last['d_loss_fake']:.3f}; G {first['g_loss']:.3f} -> {last['g_loss']:.3f}.")
print()
real_len = props_real["length"]; real_q = props_real["net_charge"]; real_h = props_real["hydrophobicity"]
print(f"Referencia real: longitud {real_len.mean():.1f} +- {real_len.std():.1f}, carga {real_q.mean():.2f} +- {real_q.std():.2f}, "
      f"GRAVY {real_h.mean():.2f} +- {real_h.std():.2f}.")
for mode in ("sample", "argmax"):
    m = fin.loc[mode]
    seqs = valid_seqs[mode]
    counts = Counter(seqs)
    top_seq, top_n = counts.most_common(1)[0] if counts else ("", 0)
    print(f"\n[{mode}] época {EPOCHS}")
    print(f"  - válidas {m['valid_fraction']:.1%}; únicas {m['unique_fraction']:.1%} ({len(counts)} distintas de {len(seqs)} válidas); "
          f"nuevas {m['novel_fraction']:.1%}. Secuencia más repetida: {top_n} veces ({top_seq[:30]}).")
    print(f"  - longitud {m['length_mean']:.1f} +- {m['length_std']:.1f} (W1 = {m['length_wasserstein']:.2f}; suelo real_subset {fl['length_wasserstein']:.2f}, random_uniform {ru['length_wasserstein']:.2f}).")
    print(f"  - carga neta {m['net_charge_mean']:.2f} +- {m['net_charge_std']:.2f} (W1 = {m['net_charge_wasserstein']:.2f}; suelos {fl['net_charge_wasserstein']:.2f} / {ru['net_charge_wasserstein']:.2f}).")
    print(f"  - GRAVY {m['hydrophobicity_mean']:.2f} +- {m['hydrophobicity_std']:.2f} (W1 = {m['hydrophobicity_wasserstein']:.2f}; suelos {fl['hydrophobicity_wasserstein']:.2f} / {ru['hydrophobicity_wasserstein']:.2f}).")
    diff = (comp[mode] - comp["real"]) * 100
    over = ", ".join(f"{aa} {v:+.1f}" for aa, v in diff.sort_values(ascending=False).head(3).items())
    under = ", ".join(f"{aa} {v:+.1f}" for aa, v in diff.sort_values().head(3).items())
    print(f"  - composición: variación total = {m['composition_tv']:.3f} (suelos {fl['composition_tv']:.3f} / {ru['composition_tv']:.3f}); "
          f"mayor exceso (puntos porcentuales): {over}; mayor defecto: {under}.")

## g. GAN vanilla estabilizada

La vanilla pura de la sección anterior colapsa: el discriminador gana desde las primeras épocas (su pérdida cae a ~1e-7) y el generador acaba produciendo pocas secuencias distintas. Una causa probable es que el discriminador distingue las secuencias reales (one-hot exacto, valores 0/1) de las generadas (probabilidades softmax, valores intermedios). Aquí se entrena, con **el mismo protocolo** (200 épocas, instantáneas cada 20, 2000 secuencias por evaluación, semilla 42), una variante con tres correcciones estándar para datos discretos. La lista es cerrada: no se cambia nada más (tasas de aprendizaje, arquitectura, normalizaciones, pasos).

1. **Suavizado unilateral de etiquetas** (`real_label = 0,9`): en el paso de D, las muestras reales llevan etiqueta 0,9 en lugar de 1; las falsas siguen en 0 y en el paso de G la etiqueta sigue siendo 1. Motivo: evitar que D se vuelva excesivamente seguro (salidas cercanas a 0 y 1), que es lo que se observa en la pura.
2. **Ruido de instancia** (desviación típica inicial 0,1, decreciendo linealmente hasta 0 en las primeras 100 épocas): se suma ruido gaussiano a las entradas de D, reales y generadas, también en el paso de G. Motivo: con ruido, las dos distribuciones se solapan y D no puede separarlas mirando solo si los valores son exactamente 0/1.
3. **Salida discreta diferenciable** (Gumbel-softmax con *straight-through*, τ = 1): entre G y D las muestras generadas se discretizan a one-hot (adelante) con gradiente suave (atrás). G sigue devolviendo probabilidades para generar y evaluar. Motivo: D ve entonces entradas del mismo tipo (one-hot exacto) en las reales y en las generadas, de modo que la diferencia entre 0/1 y valores intermedios deja de delatar a las generadas.

Criterio de «mejor» fijado de antemano: en el modo **sample**, la estabilizada es mejor que la pura si reduce la distancia de Wasserstein en las **tres** propiedades (longitud, carga e hidrofobicidad), reduce la variación total de la composición y aumenta `unique_fraction`.

### Entrenamiento de la variante estabilizada

Se usa `GANConfig.stabilized()` y el mismo `run_gan_experiment`, con la misma semilla que la pura. El tiempo por época es algo mayor que el de la pura por el ruido y el muestreo de Gumbel.

In [ ]:
config_s = GANConfig.stabilized()
print(config_s)

res_s = run_gan_experiment(X, df, config_s, epochs=EPOCHS, snapshot_every=SNAPSHOT_EVERY, n_eval=N_EVAL, eval_seed=SEED)
t_s = res_s.timing
print(f"Tiempo total: {t_s['total_s']:.1f} s (entrenamiento {t_s['train_s']:.1f} s + evaluaciones {t_s['eval_s']:.1f} s)")
print(f"Por época: media {t_s['mean_epoch_s']:.2f} s, mediana {t_s['median_epoch_s']:.2f} s")

### Curvas de pérdida: vanilla pura frente a estabilizada

Pérdidas medias por época de las dos variantes. Las dos primeras gráficas (discriminador con lotes reales y con lotes falsos) están en escala logarítmica porque la pura llega a ~1e-7; la tercera (generador) es lineal. La línea punteada es ln 2 ≈ 0,693. La línea vertical marca la época 100, en la que el ruido de instancia de la estabilizada llega a 0. Con el suavizado de etiquetas, la pérdida de D con lotes reales de la estabilizada no puede bajar de H(0,9) ≈ 0,325 aunque D acierte siempre (línea punteada naranja).

In [ ]:
h_s = res_s.history
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
panels_loss = [("d_loss_real", "Pérdida de D con lotes reales", True),
               ("d_loss_fake", "Pérdida de D con lotes falsos", True),
               ("g_loss", "Pérdida de G", False)]
for ax, (key, title, log) in zip(axes, panels_loss):
    ax.plot(epochs_axis, h[key], color="tab:red", label="vanilla pura")
    ax.plot(epochs_axis, h_s[key], color="tab:green", label="estabilizada")
    ax.axhline(np.log(2), color="gray", ls=":", lw=1, label="ln 2")
    ax.axvline(100, color="gray", ls="--", lw=0.8, label="fin del ruido (época 100)")
    if key == "d_loss_real":
        ax.axhline(-(0.9 * np.log(0.9) + 0.1 * np.log(0.1)), color="tab:orange", ls=":", lw=1, label="mínimo con etiqueta 0,9")
    if log:
        ax.set_yscale("log")
    ax.set_title(title + (" (escala log.)" if log else ""))
    ax.set_xlabel("Época")
    ax.set_ylabel("Pérdida (entropía cruzada binaria)")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

hist_s = pd.DataFrame({k: h_s[k] for k in ("d_loss_real", "d_loss_fake", "d_loss", "g_loss")}, index=epochs_axis)
display(pd.concat({
    "vanilla pura": pd.DataFrame({"primeras 5 épocas": hist_df.head(5).mean(), "últimas 5 épocas": hist_df.tail(5).mean()}),
    "estabilizada": pd.DataFrame({"primeras 5 épocas": hist_s.head(5).mean(), "últimas 5 épocas": hist_s.tail(5).mean()}),
}, axis=1).round(4))

### Métricas finales: vanilla pura, estabilizada y suelos

Métricas de la época final (200) en cada modo de decodificación, junto a los dos suelos (`real_subset`: lo mínimo alcanzable por el ruido del muestreo; `random_uniform`: ningún aprendizaje). Mismas columnas y definiciones que en la sección e.

In [ ]:
def final_table(mode):
    rows = {
        "vanilla pura": res.metrics[(res.metrics["epoch"] == EPOCHS) & (res.metrics["mode"] == mode)].iloc[0],
        "estabilizada": res_s.metrics[(res_s.metrics["epoch"] == EPOCHS) & (res_s.metrics["mode"] == mode)].iloc[0],
        "suelo real_subset": pd.Series(floors["real_subset"]),
        "suelo random_uniform": pd.Series(floors["random_uniform"]),
    }
    return pd.DataFrame(rows).T[cols]

print("Modo sample (principal), época final:")
display(final_table("sample").round(3))
print("Modo argmax (secundario), época final:")
display(final_table("argmax").round(3))

### Distribuciones: real, vanilla pura y estabilizada (época final)

Histogramas normalizados a fracción de secuencias, con los mismos intervalos y los mismos ejes para las tres series en cada panel. Arriba, modo sample; abajo, modo argmax. Solo se usan las secuencias generadas válidas. El eje vertical se ajusta a la distribución real y a la estabilizada; el pico de la vanilla pura (casi todas sus secuencias son iguales) se sale de escala y se anota con su altura.

In [ ]:
valid_seqs_s = {m: [s for s in res_s.sequences[m] if is_valid(s)] for m in ("sample", "argmax")}
prof_gen_s = {m: profile_sequences(v) if v else None for m, v in valid_seqs_s.items()}
variants = [("vanilla pura", prof_gen, "tab:red"), ("estabilizada", prof_gen_s, "tab:green")]

# Intervalos comunes por propiedad (real y las dos variantes en ambos modos).
edges_cmp = {}
for prop, _ in PROPS:
    allv = np.concatenate([prof_real.values(prop)] + [p.values(prop) for _, d, _ in variants for p in d.values() if p is not None])
    lo, hi = np.floor(allv.min()), np.ceil(allv.max())
    edges_cmp[prop] = np.arange(lo - 0.5, hi + 1.5) if prop == "length" else np.linspace(lo, hi, 41)

# Alturas de cada serie, para fijar un eje vertical común por columna a partir de la real y la estabilizada
# (el pico de una variante colapsada puede salirse de escala y se anota con su valor).
def hist_peak(values, edges):
    counts, _ = np.histogram(values, bins=edges)
    return counts.max() / len(values), edges[counts.argmax()], edges[counts.argmax() + 1]

ytop = {}
for prop, _ in PROPS:
    peaks = [hist_peak(prof_real.values(prop), edges_cmp[prop])[0]]
    peaks += [hist_peak(prof_gen_s[m].values(prop), edges_cmp[prop])[0] for m in ("sample", "argmax") if prof_gen_s[m] is not None]
    ytop[prop] = max(peaks) * 1.45

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="col", sharey="col")
for r, mode in enumerate(("sample", "argmax")):
    for c, (prop, xlabel) in enumerate(PROPS):
        ax = axes[r, c]
        real = prof_real.values(prop)
        ax.hist(real, bins=edges_cmp[prop], weights=np.full(real.size, 1 / real.size), color="tab:blue", alpha=0.4,
                label=f"real (n={real.size:,})")
        for name, d, color in variants:
            p = d[mode]
            if p is not None:
                g = p.values(prop)
                ax.hist(g, bins=edges_cmp[prop], weights=np.full(g.size, 1 / g.size), histtype="step", lw=1.6, color=color,
                        label=f"{name} (n={g.size:,})")
                peak, lo_e, hi_e = hist_peak(g, edges_cmp[prop])
                if peak > ytop[prop]:
                    ax.annotate(f"{name}: pico {peak:.2f}\n(recortado)", xy=((lo_e + hi_e) / 2, ytop[prop]),
                                xytext=(-8, -4), textcoords="offset points", ha="right", va="top", fontsize=7, color=color)
        ax.set_ylim(0, ytop[prop])
        ax.set_title(f"{xlabel} - {mode}")
        ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel("Fracción de secuencias")
        ax.legend(fontsize=7, loc="upper right")
fig.tight_layout()
plt.show()

### Composición de aminoácidos: real, vanilla pura y estabilizada

Fracción media de cada aminoácido (media de las fracciones por secuencia) en las secuencias reales y en las generadas válidas de cada variante, con el mismo orden (por abundancia en las reales) y el mismo eje vertical en los dos modos.

In [ ]:
comp_cmp = pd.DataFrame({"real": prof_real.composition}, index=list(STANDARD_ALPHABET))
for mode in ("sample", "argmax"):
    comp_cmp[f"pura_{mode}"] = prof_gen[mode].composition if prof_gen[mode] is not None else np.nan
    comp_cmp[f"estab_{mode}"] = prof_gen_s[mode].composition if prof_gen_s[mode] is not None else np.nan
comp_cmp = comp_cmp.sort_values("real", ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, sharey=True)
x = np.arange(len(comp_cmp))
w = 0.27
for ax, mode in zip(axes, ("sample", "argmax")):
    for i, (col, label, color) in enumerate((("real", "real", "tab:blue"),
                                              (f"pura_{mode}", "vanilla pura", "tab:red"),
                                              (f"estab_{mode}", "estabilizada", "tab:green"))):
        ax.bar(x + (i - 1) * w, comp_cmp[col] * 100, width=w, color=color, label=label)
    ax.set_ylabel("Composición media (%)")
    ax.set_title(f"Composición media de aminoácidos - {mode} (época final)")
    ax.legend()
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(comp_cmp.index)
axes[-1].set_xlabel("Aminoácido (ordenados por abundancia en las secuencias reales)")
fig.tight_layout()
plt.show()

### Evolución de las métricas por instantánea

Las mismas métricas que en la sección e para las dos variantes (rojo: vanilla pura; verde: estabilizada; línea continua: sample; discontinua: argmax). Las líneas horizontales son los suelos `real_subset` (gris) y, en la media de longitud, la media real.

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
for ax, (col, title, real_value) in zip(axes.ravel(), panels):
    for name, r_, color in (("vanilla pura", res, "tab:red"), ("estabilizada", res_s, "tab:green")):
        for mode, ls in (("sample", "-"), ("argmax", "--")):
            sel = r_.metrics[r_.metrics["mode"] == mode]
            ax.plot(sel["epoch"], sel[col], ls, marker="o", ms=3.5, color=color)
    ref_line = real_value if real_value is not None else floors["real_subset"][col]
    ax.axhline(ref_line, color="gray", ls=":")
    ax.set_title(title)
    ax.set_xlabel("Época")
    ax.set_ylabel(title)
handles = [Line2D([], [], color=c_, ls=ls_, marker="o", ms=3.5, label=f"{n_}, {m_}")
           for n_, c_ in (("vanilla pura", "tab:red"), ("estabilizada", "tab:green"))
           for m_, ls_ in (("sample", "-"), ("argmax", "--"))]
handles.append(Line2D([], [], color="gray", ls=":", label="suelo real_subset (media real en la longitud media)"))
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

### Secuencias de ejemplo y repetición

Las diez primeras secuencias de la variante estabilizada en cada modo (época final), con su longitud, carga neta e hidrofobicidad. Debajo, los recuentos de repetición de las dos variantes: secuencias válidas, distintas entre ellas y veces que aparece la más repetida.

In [ ]:
def example_table_s(mode, k=10):
    rows = []
    for s in res_s.sequences[mode][:k]:
        if is_valid(s):
            p = profile_sequences([s])
            rows.append({"sequence": s, "length": len(s), "net_charge": p.net_charge[0], "hydrophobicity": p.hydrophobicity[0]})
        else:
            rows.append({"sequence": s if s else "(vacía)", "length": len(s), "net_charge": np.nan, "hydrophobicity": np.nan})
    return pd.DataFrame(rows).round(2)

print("Estabilizada, modo sample:")
display(example_table_s("sample"))
print("Estabilizada, modo argmax:")
display(example_table_s("argmax"))

rep_rows = []
for name, r_ in (("vanilla pura", res), ("estabilizada", res_s)):
    for mode in ("sample", "argmax"):
        valid = [s for s in r_.sequences[mode] if is_valid(s)]
        cnt = Counter(valid)
        top_seq, top_n = cnt.most_common(1)[0] if cnt else ("", 0)
        rep_rows.append({"variante": name, "modo": mode, "generadas": len(r_.sequences[mode]), "válidas": len(valid),
                         "distintas": len(cnt), "más repetida (veces)": top_n, "secuencia más repetida": top_seq[:40]})
print("Recuentos de repetición (época final):")
display(pd.DataFrame(rep_rows).set_index(["variante", "modo"]))

### Conclusiones de la comparación (descriptivas)

Se evalúa el criterio de «mejor» fijado de antemano (modo sample, época final) con las cifras de esta ejecución. El texto se limita a decir cuáles de los cinco requisitos se cumplen y cuáles no, sin interpretar causas ni decidir el uso posterior de la variante.

In [ ]:
fin_p = res.metrics[res.metrics["epoch"] == EPOCHS].set_index("mode")
fin_s = res_s.metrics[res_s.metrics["epoch"] == EPOCHS].set_index("mode")
criteria = [
    ("length_wasserstein", "W1 de longitud", "menor"),
    ("net_charge_wasserstein", "W1 de carga neta", "menor"),
    ("hydrophobicity_wasserstein", "W1 de hidrofobicidad", "menor"),
    ("composition_tv", "variación total de composición", "menor"),
    ("unique_fraction", "unique_fraction", "mayor"),
]
print("Criterio fijado de antemano: en el modo sample, la estabilizada es mejor que la pura si reduce W1 en las tres propiedades,")
print("reduce la variación total de la composición y aumenta unique_fraction (cinco requisitos).\n")
met = []
for col, label, wanted in criteria:
    pure_v, stab_v = fin_p.loc["sample", col], fin_s.loc["sample", col]
    ok = stab_v < pure_v if wanted == "menor" else stab_v > pure_v
    met.append(ok)
    print(f"  - {label}: pura {pure_v:.3f}, estabilizada {stab_v:.3f} -> {'SE CUMPLE' if ok else 'NO se cumple'} (se pedía {wanted}).")
print(f"\nRequisitos cumplidos: {sum(met)} de {len(met)}. Según el criterio, la estabilizada "
      f"{'es' if all(met) else 'no es'} mejor que la pura.")

print()
for name, r_, hs, tt in (("vanilla pura", res, hist_df, t), ("estabilizada", res_s, hist_s, t_s)):
    f5, l5 = hs.head(5).mean(), hs.tail(5).mean()
    print(f"{name}: {tt['mean_epoch_s']:.2f} s/época; pérdidas medias primeras 5 -> últimas 5 épocas: "
          f"D real {f5['d_loss_real']:.3f} -> {l5['d_loss_real']:.3f}; D falso {f5['d_loss_fake']:.3f} -> {l5['d_loss_fake']:.3f}; "
          f"G {f5['g_loss']:.3f} -> {l5['g_loss']:.3f}.")
for name, rr in (("vanilla pura", rep_rows[:2]), ("estabilizada", rep_rows[2:])):
    print(f"{name}: " + "; ".join(f"[{d['modo']}] {d['distintas']} secuencias distintas de {d['válidas']} válidas "
                                    f"(la más repetida, {d['más repetida (veces)']} veces)" for d in rr) + ".")